# Lunar DEM Calibration Against LROC Reference Data

This notebook contains the reference-DEM preparation and elevation
calibration steps developed during the ISRO Bharatiya Antariksh
Hackathon 2025.

The workflow uses a reference DEM from LROC QuickMap and compares it
with the reconstructed lunar surface to estimate an elevation
transformation.

In [ ]:

from pathlib import Path
import imageio.v2 as imageio
import numpy as np

# --- 1) Load the reference DEM ---

reference_dem_path = Path("reference_dem.tif")

if not reference_dem_path.is_file():
    raise FileNotFoundError(
        f"Reference DEM not found: {reference_dem_path}. "
        "Place your reference DEM in the repository directory "
        "or update reference_dem_path."
    )

z_true_reference = imageio.imread(
    reference_dem_path
).astype(np.float32)

print("Reference DEM loaded successfully.")
print(f"Shape: {z_true_reference.shape}")
print(f"Data type: {z_true_reference.dtype}")
print(
    f"Elevation range: "
    f"{np.nanmin(z_true_reference):.3f} to "
    f"{np.nanmax(z_true_reference):.3f}"
)


## Load the Reconstructed Relative DEM

Load the relative DEM generated by the Shape-from-Shading notebook.

Inspect the dimensions and numerical ranges of the reconstructed
and reference DEMs before proceeding with calibration.

In [ ]:
from pathlib import Path
import imageio.v2 as imageio
import numpy as np

# Load the relative DEM generated by the SFS notebook
sfs_dem_path = Path("initial_predicted_dem.tiff")

if not sfs_dem_path.is_file():
    raise FileNotFoundError(
        f"Reconstructed DEM not found: {sfs_dem_path}. "
        "Run the SFS notebook first and save its output here."
    )

z = imageio.imread(sfs_dem_path).astype(np.float32)

# Inspect the two DEMs
print("Reconstructed DEM")
print(f"Shape: {z.shape}")
print(f"Elevation range: {np.nanmin(z):.4f} to {np.nanmax(z):.4f}")

print("\nReference DEM")
print(f"Shape: {z_true_reference.shape}")
print(
    f"Elevation range: "
    f"{np.nanmin(z_true_reference):.4f} to "
    f"{np.nanmax(z_true_reference):.4f}"
)

# Check whether direct pixel-by-pixel comparison is possible
if z.shape == z_true_reference.shape:
    print("\nThe array dimensions match.")
    print("Verify spatial alignment before fitting the calibration.")
else:
    print("\nThe array dimensions differ.")
    print("Spatial alignment or a suitable crop is required before calibration.")

## Fit Linear Elevation Calibration

Fit a linear transformation between the reconstructed relative DEM
and the reference DEM using corresponding valid pixels.

The fitted scale and offset are used to map the relative height values
onto the elevation scale of the reference DEM.

The two arrays must have matching dimensions and spatial alignment
before the regression is performed.

In [ ]:
from sklearn.linear_model import LinearRegression

# --- Fit linear calibration against the reference DEM ---

# Ensure both arrays have matching dimensions
if z.shape != z_true_reference.shape:
    raise ValueError(
        f"DEM shapes do not match: reconstructed DEM {z.shape}, "
        f"reference DEM {z_true_reference.shape}. "
        "Align or crop both arrays to the same grid before calibration."
    )

# Use only pixels with valid values in both DEMs
valid_mask = np.isfinite(z) & np.isfinite(z_true_reference)

if np.count_nonzero(valid_mask) < 2:
    raise ValueError("Insufficient valid pixels for linear calibration.")

# Fit reference elevation = a * reconstructed height + b
z_final_flat = z[valid_mask].reshape(-1, 1)
z_final_true_flat = z_true_reference[valid_mask]

reg = LinearRegression().fit(
    z_final_flat,
    z_final_true_flat
)

a = float(reg.coef_[0])
b = float(reg.intercept_)

print(f"z_abs = {a:.6g} * z + {b:.6g}")

# Apply calibration to the reconstructed DEM
z_final_abs = (a * z + b).astype(np.float32)

## Save and Visualize the Calibrated DEM

Save the calibrated elevation map as a floating-point TIFF file.

Compare the calibrated reconstruction with the reference DEM using
side-by-side elevation maps with a shared colour scale.

In [ ]:
import tifffile
import matplotlib.pyplot as plt

# --- Save the calibrated DEM ---

output_path = "predicted_final_absolute_dem.tiff"

tifffile.imwrite(
    output_path,
    z_final_abs.astype(np.float32)
)

print(f"Calibrated DEM saved to: {output_path}")

# --- Compare the calibrated and reference DEMs ---

# Use common valid pixels to define a shared colour scale
comparison_values = np.concatenate([
    z_final_abs[valid_mask],
    z_true_reference[valid_mask]
])

vmin = np.min(comparison_values)
vmax = np.max(comparison_values)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(
    np.ma.masked_invalid(z_final_abs),
    cmap="terrain",
    vmin=vmin,
    vmax=vmax
)
axes[0].set_title("Predicted Absolute DEM")
axes[0].axis("off")

im1 = axes[1].imshow(
    np.ma.masked_invalid(z_true_reference),
    cmap="terrain",
    vmin=vmin,
    vmax=vmax
)
axes[1].set_title("Reference DEM")
axes[1].axis("off")

fig.colorbar(
    im1,
    ax=axes,
    label="Elevation (reference DEM units)",
    shrink=0.85
)

plt.tight_layout()
plt.savefig(
    "dem_calibration_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()